#  Hindi STT Fine-Tuning with Whisper

**MySivi AI Engineer Intern Assignment**

**Fine-tuned model output:** https://huggingface.co/rishii100/whisper-small-hindi  
(I uploaded my results in my Huggingspace Hub)


## 1. Setup & Installation


In [ ]:
from huggingface_hub import login

# Your HuggingFace token
login(token="hf_Z.....................g")# I deleted this before sharing

print(" Logged in to HuggingFace!")

✅ Logged in to HuggingFace!


## 1b. HuggingFace Login


In [ ]:
from huggingface_hub import login

login()

print(" Logged in to HuggingFace!")

✅ Logged in to HuggingFace!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## 2. Imports & GPU Check

In [ ]:
# Step 1: Install dependencies
# ⚠️ AFTER this cell runs, go to Runtime → Restart Session, then run the NEXT cell
!pip install -q transformers[torch] "datasets<3.0.0" evaluate jiwer librosa huggingface_hub "accelerate>=0.21.0" tensorboard

print(" Packages installed!")
print("  NOW go to Runtime → Restart Session, then run the next cell")

✅ Packages installed!
⚠️  NOW go to Runtime → Restart Session, then run the next cell


In [ ]:
# Step 2: Imports (run this AFTER restarting the session)
import torch
import numpy as np
import evaluate
import gc
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_dataset, DatasetDict, Audio
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

import datasets
print(f" datasets version: {datasets.__version__}")

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"  Using device: {device}")
if device == "cuda":
    print(f" GPU: {torch.cuda.get_device_name(0)}")
    print(f" GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

📦 datasets version: 2.21.0
🖥️  Using device: cuda
🎮 GPU: Tesla T4
💾 GPU Memory: 15.6 GB


## 3. Configuration


In [ ]:
# ==================== MODIFY THESE AS NEEDED ====================
MODEL_NAME = "openai/whisper-small"          # Base model to fine-tune
LANGUAGE = "hi"                               # Hindi language code
TASK = "transcribe"                           # Task type
DATASET_NAME = "google/fleurs"                # Dataset (freely available)
DATASET_LANGUAGE = "hi_in"                    # Hindi config in FLEURS

# Training hyperparameters
BATCH_SIZE = 16                               # Per device batch size
GRADIENT_ACCUMULATION = 1                     # Gradient accumulation steps
LEARNING_RATE = 1e-5                          # Learning rate
WARMUP_STEPS = 250                            # Warmup steps
MAX_STEPS = 2000                              # Total training steps
FP16 = True                                   # Mixed precision training

# Data configuration
MAX_TRAIN_SAMPLES = None                      # Set to int to limit (e.g., 5000)
MAX_EVAL_SAMPLES = None                       # Set to int to limit (e.g., 500)
MAX_TEST_SAMPLES = None                       # Set to int to limit (e.g., 500)

OUTPUT_DIR = "./whisper-small-hindi"
# =================================================================

print(" Configuration:")
print(f"   Model: {MODEL_NAME}")
print(f"   Dataset: {DATASET_NAME} ({DATASET_LANGUAGE})")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Max Steps: {MAX_STEPS}")
print(f"   Learning Rate: {LEARNING_RATE}")

📋 Configuration:
   Model: openai/whisper-small
   Dataset: google/fleurs (hi_in)
   Batch Size: 16
   Max Steps: 2000
   Learning Rate: 1e-05


## 4. Load Dataset

### Dataset: Google FLEURS (Hindi)


**Source:** https://huggingface.co/datasets/google/fleurs

In [ ]:
print(" Loading Google FLEURS Hindi dataset...")
print("   (This may take a few minutes on first run)")

# Load the dataset splits
# FLEURS uses 'transcription' field instead of 'sentence'
fleurs = load_dataset(DATASET_NAME, DATASET_LANGUAGE, trust_remote_code=True)

# Create our working dataset dict
common_voice = DatasetDict()
common_voice["train"] = fleurs["train"]
common_voice["validation"] = fleurs["validation"]
common_voice["test"] = fleurs["test"]

print(f"\n Dataset Statistics:")
print(f"   Train:      {len(common_voice['train']):,} samples")
print(f"   Validation: {len(common_voice['validation']):,} samples")
print(f"   Test:       {len(common_voice['test']):,} samples")

# Optionally limit samples for faster runs
if MAX_TRAIN_SAMPLES:
    common_voice["train"] = common_voice["train"].select(range(MAX_TRAIN_SAMPLES))
    print(f"     Limited train to {MAX_TRAIN_SAMPLES} samples")
if MAX_EVAL_SAMPLES:
    common_voice["validation"] = common_voice["validation"].select(range(MAX_EVAL_SAMPLES))
    print(f"     Limited validation to {MAX_EVAL_SAMPLES} samples")
if MAX_TEST_SAMPLES:
    common_voice["test"] = common_voice["test"].select(range(MAX_TEST_SAMPLES))
    print(f"     Limited test to {MAX_TEST_SAMPLES} samples")

📥 Loading Google FLEURS Hindi dataset...
   (This may take a few minutes on first run)

📊 Dataset Statistics:
   Train:      2,120 samples
   Validation: 239 samples
   Test:       418 samples


### Explore a Sample

In [ ]:
sample = common_voice["train"][0]
print("🔍 Sample entry keys:", list(sample.keys()))
print(f"   Transcription: {sample['transcription']}")
print(f"   Audio sampling rate: {sample['audio']['sampling_rate']} Hz")
print(f"   Audio array length: {len(sample['audio']['array'])} samples")
print(f"   Duration: {len(sample['audio']['array']) / sample['audio']['sampling_rate']:.2f} seconds")

🔍 Sample entry keys: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id']
   Transcription: राजनीतिज्ञों ने कहा कि उन्होंने निर्णायक मत को अनावश्यक रूप से निर्धारित करने के लिए अफ़गान संविधान में काफी अस्पष्टता पाई थी
   Audio sampling rate: 16000 Hz
   Audio array length: 138240 samples
   Duration: 8.64 seconds


## 5. Load Whisper Model Components

### Model: OpenAI Whisper Small (244M parameters)



In [ ]:
print(" Loading Whisper components...")

# Feature extractor: converts audio → log-Mel spectrogram
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)

# Tokenizer: converts text → token IDs (Hindi)
tokenizer = WhisperTokenizer.from_pretrained(
    MODEL_NAME,
    language=LANGUAGE,
    task=TASK
)

# Processor: combines feature extractor + tokenizer
processor = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    language=LANGUAGE,
    task=TASK
)

print(" Feature extractor, tokenizer, and processor loaded!")

📦 Loading Whisper components...
✅ Feature extractor, tokenizer, and processor loaded!


In [ ]:
print("🤖 Loading Whisper Small model...")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

# Configure for Hindi transcription
model.generation_config.language = LANGUAGE
model.generation_config.task = TASK
model.generation_config.forced_decoder_ids = None

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f" Model loaded!")
print(f"   Total parameters: {total_params / 1e6:.1f}M")
print(f"   Trainable parameters: {trainable_params / 1e6:.1f}M")

🤖 Loading Whisper Small model...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

✅ Model loaded!
   Total parameters: 241.7M
   Trainable parameters: 241.7M


## 6. Data Preprocessing

### Pipeline:
```
Audio (various sample rates) → Resample to 16kHz → Log-Mel Spectrogram
Text (Hindi sentences) → Whisper Tokenizer → Token IDs
```

In [ ]:
# Ensure all audio is resampled to 16kHz (Whisper's requirement)
common_voice = common_voice.cast_column("audio", Audio(sampling_rate=16000))

# Keep only the columns we need (FLEURS uses 'transcription' not 'sentence')
common_voice = common_voice.remove_columns(
    [col for col in common_voice["train"].column_names
     if col not in ["audio", "transcription"]]
)

print(" Audio resampled to 16kHz")
print(f" Columns kept: {common_voice['train'].column_names}")

🎵 Audio resampled to 16kHz
📝 Columns kept: ['audio', 'transcription']


In [ ]:
def prepare_dataset(batch):
    """
    Preprocess a batch of audio-text pairs for Whisper:
    1. Extract log-Mel spectrogram features from audio
    2. Tokenize the target text (Hindi transcription)
    """
    audio = batch["audio"]

    # Compute log-Mel spectrogram features
    batch["input_features"] = feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    # FLEURS uses 'transcription' field (not 'sentence')
    batch["labels"] = tokenizer(batch["transcription"]).input_ids

    return batch


print(" Preprocessing datasets (this may take a while)...")

common_voice = common_voice.map(
    prepare_dataset,
    remove_columns=common_voice.column_names["train"],
    num_proc=1,
)

print(f" Preprocessing complete!")
print(f"   Train features: {len(common_voice['train'])}")
print(f"   Validation features: {len(common_voice['validation'])}")
print(f"   Test features: {len(common_voice['test'])}")

⏳ Preprocessing datasets (this may take a while)...
✅ Preprocessing complete!
   Train features: 2120
   Validation features: 239
   Test features: 418


## 7. Data Collator



In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """
    Custom data collator for Whisper fine-tuning.
    Handles padding of input features and labels.
    Labels are padded with -100 (ignored in loss computation).
    """
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

print(" Data collator ready!")

✅ Data collator ready!


## 8. Evaluation Metrics

- **WER (Word Error Rate)**: (Substitutions + Insertions + Deletions) / Total Words × 100
- **CER (Character Error Rate)**: Character-level accuracy

In [ ]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


def compute_metrics(pred):
    """Compute WER and CER for evaluation during training."""
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    cer = 100 * cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}


print(" Evaluation metrics configured (WER + CER)")

✅ Evaluation metrics configured (WER + CER)


## 9. Training Configuration


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    max_steps=MAX_STEPS,
    gradient_checkpointing=True,
    fp16=FP16,
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=400,
    eval_steps=400,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
    save_total_limit=3,
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

print(" Training arguments configured!")

✅ Training arguments configured!


In [ ]:
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=common_voice["train"],
    eval_dataset=common_voice["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

print(" Trainer initialized!")
print("=" * 60)
print("🚀 Run the next cell to start training!")
print("   Estimated time: ~2-3 hours on T4 GPU")
print("=" * 60)

✅ Trainer initialized!
🚀 Run the next cell to start training!
   Estimated time: ~2-3 hours on T4 GPU


## 10. Fine-Tuning



In [ ]:
print(" Starting fine-tuning...")
print("=" * 60)

train_result = trainer.train()

print("\n" + "=" * 60)
print(" Training complete!")
print("=" * 60)
print(f"   Training loss: {train_result.training_loss:.4f}")
print(f"   Training runtime: {train_result.metrics['train_runtime']:.1f} seconds")
print(f"   Samples/second: {train_result.metrics['train_samples_per_second']:.1f}")

# Save the final model
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"\n Model saved to {OUTPUT_DIR}")

🚀 Starting fine-tuning...


Step,Training Loss,Validation Loss,Wer,Cer
400,0.144165,0.278299,29.851024,11.430264
800,0.025790,0.335559,27.746741,10.756589
1200,0.002626,0.393425,27.895717,11.004590
1600,0.000753,0.421301,27.094972,10.490080
2000,0.000488,0.432104,27.206704,10.482677


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].



🎉 Training complete!
   Training loss: 0.0796
   Training runtime: 6887.3 seconds
   Samples/second: 4.6


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


💾 Model saved to ./whisper-small-hindi


## 11. Evaluation

We evaluate **both** the base and fine-tuned models on the **same held-out test set** that was **never** used during training.

### 11a. Evaluate Base Model (Before Fine-Tuning)

In [21]:
print("=" * 60)
print(" Evaluating BASE model on test set...")
print("=" * 60)

base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
base_model.generation_config.language = LANGUAGE
base_model.generation_config.task = TASK
base_model.generation_config.forced_decoder_ids = None
base_model = base_model.float()
base_model.to(device)
base_model.eval()

base_predictions = []
base_references = []

test_dataset = common_voice["test"]
eval_batch_size = 8

print(f"   Evaluating {len(test_dataset)} test samples...")

for i in range(0, len(test_dataset), eval_batch_size):
    batch_end = min(i + eval_batch_size, len(test_dataset))
    batch_indices = list(range(i, batch_end))

    input_features_list = [test_dataset[idx]["input_features"] for idx in batch_indices]
    label_ids_list = [test_dataset[idx]["labels"] for idx in batch_indices]

    input_features = torch.tensor(np.array(input_features_list), dtype=torch.float32).to(device)

    with torch.no_grad():
        predicted_ids = base_model.generate(input_features, max_length=225)

    pred_text = tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)

    for label_ids in label_ids_list:
        label_ids_cleaned = [l for l in label_ids if l != -100]
        ref_text = tokenizer.decode(label_ids_cleaned, skip_special_tokens=True)
        base_references.append(ref_text)

    base_predictions.extend(pred_text)

    if (i // eval_batch_size) % 10 == 0:
        print(f"   Progress: {batch_end}/{len(test_dataset)}")

base_wer = 100 * wer_metric.compute(predictions=base_predictions, references=base_references)
base_cer = 100 * cer_metric.compute(predictions=base_predictions, references=base_references)

print(f"\n Base Model Results:")
print(f"   WER: {base_wer:.2f}%")
print(f"   CER: {base_cer:.2f}%")

del base_model
gc.collect()
torch.cuda.empty_cache()


📊 Evaluating BASE model on test set...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

   Evaluating 418 test samples...
   Progress: 8/418
   Progress: 88/418
   Progress: 168/418
   Progress: 248/418
   Progress: 328/418
   Progress: 408/418

📊 Base Model Results:
   WER: 68.61%
   CER: 34.43%


### 11b. Evaluate Fine-Tuned Model

In [23]:
print("=" * 60)
print(" Evaluating FINE-TUNED model on test set...")
print("=" * 60)

ft_predictions = []
ft_references = base_references.copy()

model = model.float()
model.eval()
model.to(device)

for i in range(0, len(test_dataset), eval_batch_size):
    batch_end = min(i + eval_batch_size, len(test_dataset))
    batch_indices = list(range(i, batch_end))

    input_features_list = [test_dataset[idx]["input_features"] for idx in batch_indices]
    input_features = torch.tensor(np.array(input_features_list), dtype=torch.float32).to(device)

    with torch.no_grad():
        predicted_ids = model.generate(input_features, max_length=225)

    pred_text = tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)
    ft_predictions.extend(pred_text)

    if (i // eval_batch_size) % 10 == 0:
        print(f"   Progress: {batch_end}/{len(test_dataset)}")

ft_wer = 100 * wer_metric.compute(predictions=ft_predictions, references=ft_references)
ft_cer = 100 * cer_metric.compute(predictions=ft_predictions, references=ft_references)

print(f"\n Fine-Tuned Model Results:")
print(f"   WER: {ft_wer:.2f}%")
print(f"   CER: {ft_cer:.2f}%")


📊 Evaluating FINE-TUNED model on test set...
   Progress: 8/418
   Progress: 88/418
   Progress: 168/418
   Progress: 248/418
   Progress: 328/418
   Progress: 408/418

📊 Fine-Tuned Model Results:
   WER: 26.87%
   CER: 10.41%


## 12. Results Comparison

In [25]:
print("\n" + "=" * 60)
print(" RESULTS COMPARISON")
print("=" * 60)
print(f"{'Model':<28} {'WER (%)':>10} {'CER (%)':>10}")
print("-" * 50)
print(f"{'Whisper Small (Base)':<28} {base_wer:>10.2f} {base_cer:>10.2f}")
print(f"{'Whisper Small (Fine-tuned)':<28} {ft_wer:>10.2f} {ft_cer:>10.2f}")
print("-" * 50)

wer_improvement = base_wer - ft_wer
cer_improvement = base_cer - ft_cer

print(f"{'Improvement':<28} {wer_improvement:>+10.2f} {cer_improvement:>+10.2f}")
print(f"{'Relative Improvement':<28} {(wer_improvement/base_wer)*100:>+9.1f}% {(cer_improvement/base_cer)*100:>+9.1f}%")
print("=" * 50)

if wer_improvement > 0:
    print(f"\n Fine-tuning improved WER by {wer_improvement:.2f} percentage points!")
else:
    print(f"\n  Fine-tuning did not improve WER. Consider more training steps or data.")


🏆 RESULTS COMPARISON
Model                           WER (%)    CER (%)
--------------------------------------------------
Whisper Small (Base)              68.61      34.43
Whisper Small (Fine-tuned)        26.87      10.41
--------------------------------------------------
Improvement                      +41.73     +24.02
Relative Improvement             +60.8%     +69.8%

 Fine-tuning improved WER by 41.73 percentage points!


## 13. Error Analysis — Sample Comparisons

In [27]:
import random
random.seed(42)
sample_indices = random.sample(range(len(ft_references)), min(10, len(ft_references)))

for idx in sample_indices:
    print(f"\n--- Sample {idx} ---")
    print(f" Reference:   {ft_references[idx]}")
    print(f" Base Model:  {base_predictions[idx]}")
    print(f" Fine-Tuned:  {ft_predictions[idx]}")


--- Sample 327 ---
 Reference:   ऐसा माना जाता है कि पेरिस के निवासी अहंकारी असभ्य और अभिमानी होते हैं
 Base Model:   आईसा माना जाता है कि पैरिस की निवासी एंकारी असवब़ और अबिमानी होते है
 Fine-Tuned:  ऐसा माना जाता है कि पेरिस की निवासी अयंकारी असब्ब्व और अबिमानी होते हैं

--- Sample 57 ---
 Reference:   महिलाएं यह अनुशंसा की जाती है कि कोई भी महिला यात्री वास्तविक वैवाहिक स्थिति के बावजूद कहती है कि वह विवाहित है
 Base Model:   मैलाय यह अनसुन्सा की जाती है के कोई भी मैलायात्री वास्तवे के वेवाये के जिस्तिके बाजुद कैती है के वेवायेत है.
 Fine-Tuned:  महिलाएं यह अनसूनसा की जाती है कि कोई भी महिला यात्री वास्तविक व्यवाय के सित्थी के बावजूद कहती है कि वह व्यवायट है

--- Sample 12 ---
 Reference:   ms बीमारी केंद्रीय तंत्रिका तंत्र पर असर करती है जिसमें दिमाग स्पाइनल कॉर्ड और ऑप्टिक नर्व शामिल हैं
 Base Model:   ms बीमारी के अंदरी तन्तर का तन्तर पर असर करती है जिस में दिमांग इस पाइनल कोड और अप्टिक नर्व शामिल हैं
 Fine-Tuned:  एमएस बीमारी केंद्रीय तंत्र का तंत्र पर असर करती है जिसमें दिमांग

## 14. Transcribe Hindi audio file with the fine-tuned model


In [28]:
import librosa

def transcribe_hindi(audio_path, model, processor):
    """Transcribe a Hindi audio file using the fine-tuned Whisper model."""
    audio, sr = librosa.load(audio_path, sr=16000)

    input_features = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.float().to(device)

    with torch.no_grad():
        predicted_ids = model.generate(input_features, max_length=225)

    transcription = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    return transcription


# Test with a sample from the test set
print(" Testing with a sample from the test set...")
test_sample = common_voice["test"][0]
input_features = torch.tensor(
    np.array([test_sample["input_features"]]), dtype=torch.float32
).to(device)

with torch.no_grad():
    predicted_ids = model.generate(input_features, max_length=225)

transcription = tokenizer.decode(predicted_ids[0], skip_special_tokens=True)
reference = tokenizer.decode(
    [l for l in test_sample["labels"] if l != -100],
    skip_special_tokens=True
)

print(f" Reference:     {reference}")
print(f" Transcription: {transcription}")


🎤 Testing with a sample from the test set...
📝 Reference:     कुछ अणुओं में अस्थिर केंद्रक होता है जिसका मतलब यह है कि उनमें थोड़े या बिना किसी झटके से टूटने की प्रवृत्ति होती है
🎤 Transcription: कुछ अढ़ंघों में अस्त केन्रक होता है जिसका मतलब यहां कि उनमें थोड़ें या बिना किसी झटके से टूटने की प्रवत्ति होती है


### Upload Hindi Audio



In [29]:
from google.colab import files
import IPython.display as ipd

print(" Upload a Hindi audio file (wav, mp3, flac):")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\n Playing audio: {filename}")
    ipd.display(ipd.Audio(filename))

    print(" Transcribing...")
    result = transcribe_hindi(filename, model, processor)
    print(f" Transcription: {result}")

📤 Upload a Hindi audio file (wav, mp3, flac):


Saving 10013.mp3 to 10013.mp3

🎵 Playing audio: 10013.mp3


⏳ Transcribing...
🎤 Transcription: शाह से जोड़ा तुम स्वयन के भीतर जितना चढ़ोगे उतना ही अधिक देख पाओगे


## 15. Save Results

In [30]:
results_text = f"""
Hindi Speech-to-Text Fine-Tuning Results
==========================================

Model: {MODEL_NAME}
Dataset: {DATASET_NAME} ({DATASET_LANGUAGE})
Training Steps: {MAX_STEPS}
Learning Rate: {LEARNING_RATE}
Batch Size: {BATCH_SIZE}

Results on Test Set ({len(test_dataset)} samples):
--------------------------------------------------
Model                      WER (%)    CER (%)
Whisper Small (Base)       {base_wer:.2f}     {base_cer:.2f}
Whisper Small (Fine-tuned) {ft_wer:.2f}     {ft_cer:.2f}
--------------------------------------------------
Improvement                {wer_improvement:+.2f}     {cer_improvement:+.2f}
"""

with open("results.txt", "w") as f:
    f.write(results_text)

print(" Results saved to results.txt")

 Results saved to results.txt
